# 03 – Palepus Variablen

Auf der gefilterten Stichprobe aus Notebook 02 (33.428 Firmenjahre) bauen wir die Variablen, die Palepu (1986) für seine sechs Hypothesen verwendet:

| Hypothese (Palepu) | Variable hier |
|---|---|
| Ineffizientes Management | `roe` (Eigenkapitalrendite) |
| Wachstums-Ressourcen-Ungleichgewicht | `sales_growth_1y`, `liquidity`, `leverage`, `grdummy` |
| Branchenstörung | `idummy` |
| Größe | `log_assets` |
| Unterbewertung | `float_to_book` |
| Kurs-Gewinn-Verhältnis | `float_to_earnings` |

Hier werden nur Variablen gebaut und beschrieben – **keine Modelle**.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"
Y = "y_pe_main_h3"

panel = pd.read_csv(DATA / "processed" / "firm_year_panel.csv", dtype={"cik": str})
ratios = pd.read_csv(DATA / "processed" / "firm_year_ratios.csv", dtype={"cik": str})
fund = pd.read_csv(DATA / "processed" / "firm_year_fundamentals.csv", dtype={"cik": str})

# Verbinden wie in 01/02; aus den Fundamentaldaten die Rohwerte für roe und liquidity dazunehmen
ratio_cols = ["sales_growth_1y", "leverage", "log_assets", "float_to_book", "float_to_earnings"]
fund_cols = ["net_income", "equity", "cash", "liabilities_current", "total_assets"]
df = (panel.merge(ratios[["accession"] + ratio_cols], on="accession", validate="one_to_one")
           .merge(fund[["accession"] + fund_cols], on="accession", validate="one_to_one"))

# Stichprobenfilter exakt wie in Notebook 02 (früher archive/src/16_baseline_logit.py, jetzt nur in der Git-Historie)
s = df[df["h3_observed"]]
s = s[~s["fin_sic_60_64"].astype(bool)]
s = s[s["completion_status"] != "uncertain"]
s = s[~pd.to_numeric(s["sic"], errors="coerce").isin([6770])]
s = s[(s["entity_public_float"] >= 50e6) | s["first_listing_year_exempt"].astype(bool)]
sample = s.copy()

print(f"Stichprobe: {len(sample):,} Zeilen, {int(sample[Y].sum())} Positive")
assert len(sample) == 33_428 and sample[Y].sum() == 282

Stichprobe: 33,428 Zeilen, 282 Positive


## 1. ROE = Nettoergebnis / Eigenkapital

Bei Eigenkapital ≤ 0 ist die Kennzahl sinnlos (ein Verlust geteilt durch negatives Eigenkapital ergäbe eine *positive* Rendite), daher bleibt sie dann leer.

In [2]:
equity_pos = sample["equity"].where(sample["equity"] > 0)     # Eigenkapital <= 0 -> leer
sample["roe"] = sample["net_income"] / equity_pos

print(f"Eigenkapital <= 0: {(sample['equity'] <= 0).sum():,} Zeilen -> roe leer")
print(f"roe fehlt insgesamt: {sample['roe'].isna().mean():.1%}")

Eigenkapital <= 0: 2,769 Zeilen -> roe leer
roe fehlt insgesamt: 11.8%


## 2. Liquidität = (Cash − kurzfristige Verbindlichkeiten) / Bilanzsumme

Näherung an Palepus Definition: Wertpapiere (marketable securities) fehlen in unseren Rohdaten, daher nur Cash. Das ist eine dokumentierte Abweichung.

In [3]:
ta_pos = sample["total_assets"].where(sample["total_assets"] > 0)
sample["liquidity"] = (sample["cash"] - sample["liabilities_current"]) / ta_pos
print(f"liquidity fehlt: {sample['liquidity'].isna().mean():.1%}")
sample["liquidity"].describe().round(3)

liquidity fehlt: 11.3%


count     29634.000
mean        -15.269
std        1446.654
min     -235128.000
25%          -0.169
50%          -0.071
75%           0.046
max           0.999
Name: liquidity, dtype: float64

## 3. GRDUMMY – Ungleichgewicht zwischen Wachstum und Ressourcen

Palepus Idee: Ein Ziel ist eine Firma, bei der Wachstum und Mittel nicht zusammenpassen – entweder **wenig Wachstum, aber viel Geld** (hohe Liquidität, wenig Schulden) oder **viel Wachstum, aber wenig Geld** (niedrige Liquidität, viele Schulden). Ein Käufer kann dieses Ungleichgewicht ausgleichen.

„Hoch“ und „niedrig“ heißt: über bzw. höchstens am Median aller Stichprobenfirmen **im selben Geschäftsjahr** (der Median wird pro Jahr nur aus den vorhandenen Werten berechnet). Fehlt eine der drei Zutaten, bleibt `grdummy` leer.

In [4]:
def high_vs_year_median(col):
    """True = über dem Median des Geschäftsjahres, False = höchstens Median, NaN = Wert fehlt."""
    year_median = sample.groupby("fiscal_year")[col].transform("median")
    high = sample[col] > year_median
    return high.where(sample[col].notna())            # fehlender Wert -> NaN statt False

g_high = high_vs_year_median("sales_growth_1y")
l_high = high_vs_year_median("liquidity")
v_high = high_vs_year_median("leverage")

# Kombination 1: wenig Wachstum, viel Liquidität, wenig Schulden
combo1 = (g_high == False) & (l_high == True) & (v_high == False)
# Kombination 2: viel Wachstum, wenig Liquidität, viele Schulden
combo2 = (g_high == True) & (l_high == False) & (v_high == True)

all_present = g_high.notna() & l_high.notna() & v_high.notna()
sample["grdummy"] = (combo1 | combo2).astype(float).where(all_present)

print(f"grdummy fehlt: {sample['grdummy'].isna().mean():.1%}")
print(f"grdummy = 1 (von den vorhandenen): {sample['grdummy'].mean():.1%}")
print(f"  davon Kombination 1 (wenig Wachstum/viel Geld): {combo1.sum():,}")
print(f"  davon Kombination 2 (viel Wachstum/wenig Geld): {combo2.sum():,}")

grdummy fehlt: 33.6%
grdummy = 1 (von den vorhandenen): 26.6%
  davon Kombination 1 (wenig Wachstum/viel Geld): 2,756
  davon Kombination 2 (viel Wachstum/wenig Geld): 3,143


## 4. IDUMMY – Take-Private-Aktivität in der Branche im Vorjahr

`idummy = 1`, wenn in derselben 2-stelligen SIC-Branche im **Kalenderjahr vor** dem Geschäftsjahr mindestens ein abgeschlossenes PE-Take-Private (pe_main) stattfand. Gezählt werden alle 183 Events des Event-Universums – auch die von Firmen, die selbst nicht in der Stichprobe sind (z. B. zu klein).

**Welches Datum?** Wir nehmen `first_event_date` – den Tag der ersten öffentlichen Meldung des Deals bei der SEC (Proxy oder SC 13E-3), also faktisch die Ankündigung. Das ist dasselbe Datum, mit dem auch unsere Labels gebildet werden, und es liegt bei einem Ereignis im Vorjahr immer vor dem Bilanzstichtag des aktuellen Geschäftsjahres – es wird also kein Wissen aus der Zukunft verwendet.

**Branche der Event-Firmen:** aus den SEC-DERA-Daten (`stage_11_raw_quarters.csv`, 10-K-Meldungen ab 2010), jeweils der SIC-Code aus der letzten Meldung vor dem Event.

In [5]:
events = pd.read_csv(DATA / "interim" / "combined_events.csv", dtype={"cik": str})
events = events[events["sponsor_like_ptp"].astype(str) == "True"].copy()          # pe_main-Events
events["event_date"] = pd.to_datetime(events["first_event_date"])
events["event_year"] = events["event_date"].dt.year
print(f"pe_main-Events: {len(events)} (alle abgeschlossen: {(events['completion_status'] == 'completed').all()})")

# SIC der Event-Firma: letzte DERA-Meldung vor dem Event (sonst die früheste danach)
filings = pd.read_csv(DATA / "interim" / "stage_11_raw_quarters.csv", dtype={"cik": str},
                      usecols=["cik", "sic", "filed_date"]).dropna(subset=["sic"])
filings["filed_date"] = pd.to_datetime(filings["filed_date"])
f = filings.merge(events[["cik", "event_date"]], on="cik")
f["before"] = f["filed_date"] <= f["event_date"]
f = f.sort_values(["cik", "before", "filed_date"], ascending=[True, False, False])  # vorher-Meldungen zuerst, neueste oben
f["gap"] = (f["filed_date"] - f["event_date"]).abs()
f = pd.concat([f[f["before"]].groupby("cik").head(1),
               f[~f["before"]].sort_values("gap").groupby("cik").head(1)]).drop_duplicates("cik")
events = events.merge(f[["cik", "sic"]], on="cik", how="left")
events["sic2"] = events["sic"] // 100

no_sic = events[events["sic2"].isna()]
print(f"Events ohne SIC-Code: {len(no_sic)}; davon in 2011-2021 (relevant für FY2012-2022): "
      f"{no_sic['event_year'].between(2011, 2021).sum()}")

pe_main-Events: 183 (alle abgeschlossen: True)
Events ohne SIC-Code: 33; davon in 2011-2021 (relevant für FY2012-2022): 14


In [6]:
# Welche (Branche, Jahr)-Kombinationen hatten mindestens ein Event?
active = events.dropna(subset=["sic2"]).groupby(["sic2", "event_year"]).size().rename("n_events").reset_index()

# Für jede Stichprobenzeile: gab es in ihrer Branche ein Event im Kalenderjahr fiscal_year - 1?
sample["sic2"] = pd.to_numeric(sample["sic"], errors="coerce") // 100
sample["lookup_year"] = sample["fiscal_year"] - 1          # nur Vergangenheit
sample = sample.merge(active.rename(columns={"event_year": "lookup_year"}),
                      on=["sic2", "lookup_year"], how="left")
sample["idummy"] = (sample["n_events"].fillna(0) > 0).astype(float).where(sample["sic2"].notna())

assert (sample["lookup_year"] < sample["fiscal_year"]).all()     # Kontrolle: nie dasselbe oder ein späteres Jahr
print(f"idummy = 1: {sample['idummy'].mean():.1%} der Zeilen; fehlt: {sample['idummy'].isna().mean():.1%}")
print("Anteil idummy = 1 pro Geschäftsjahr:")
print(sample.groupby("fiscal_year")["idummy"].mean().map("{:.1%}".format).to_string())

idummy = 1: 30.1% der Zeilen; fehlt: 0.1%
Anteil idummy = 1 pro Geschäftsjahr:
fiscal_year
2012    12.9%
2013    30.5%
2014    47.0%
2015    15.0%
2016    32.3%
2017    19.3%
2018    27.9%
2019    33.5%
2020    45.9%
2021    47.2%
2022    17.4%


## 5. Übersicht: fehlende Werte und Lage, Positive vs. Negative

Bei stetigen Variablen der Median, bei den Dummies (`grdummy`, `idummy`) der Anteil der 1er.

In [7]:
palepu = ["roe", "sales_growth_1y", "liquidity", "leverage", "grdummy", "idummy",
          "log_assets", "float_to_book", "float_to_earnings"]
dummies = {"grdummy", "idummy"}

pos, neg = sample[sample[Y] == 1], sample[sample[Y] == 0]
rows = []
for v in palepu:
    stat = "Anteil 1er" if v in dummies else "Median"
    center = (lambda x: x.mean()) if v in dummies else (lambda x: x.median())
    rows.append({
        "Variable": v,
        "fehlt: Positive": f"{pos[v].isna().mean():.1%}",
        "fehlt: Negative": f"{neg[v].isna().mean():.1%}",
        "Kennwert": stat,
        "Positive": round(center(pos[v]), 3),
        "Negative": round(center(neg[v]), 3),
    })
pd.DataFrame(rows).set_index("Variable")

,fehlt: Positive,fehlt: Negative,Kennwert,Positive,Negative
Variable,,,,,
roe,16.7%,11.8%,Median,0.032,0.056
sales_growth_1y,17.7%,25.7%,Median,0.080,0.069
liquidity,8.5%,11.4%,Median,-0.099,-0.071
leverage,2.1%,2.2%,Median,0.264,0.201
grdummy,24.8%,33.7%,Anteil 1er,0.165,0.267
idummy,0.0%,0.1%,Anteil 1er,0.404,0.300
log_assets,1.1%,0.8%,Median,20.800,20.762
float_to_book,13.1%,7.2%,Median,1.694,1.983
float_to_earnings,5.3%,5.2%,Median,8.901,9.123


**Ergebnis:** Spätere Ziele haben eine **niedrigere Eigenkapitalrendite** (3,2 % gegen 5,6 %), **weniger Liquidität**, **mehr Schulden** und eine **niedrigere Bewertung** (Float/Buchwert 1,7 gegen 2,0); bei der Größe gibt es keinen Unterschied. Die beiden Dummies verhalten sich **umgekehrt zu Palepus Ergebnissen**: `grdummy` ist bei Zielen *seltener* 1 (17 % gegen 27 %; bei Palepu positiv), `idummy` dagegen *häufiger* (40 % gegen 30 %; bei Palepu negativ).

Zu den Lücken: Bei den Positiven fehlen `roe` und `float_to_book` öfter, weil vor allem **Partnerships (MLPs, z. B. Buckeye Partners, Tallgrass)** ihr Eigenkapital als „partners' capital“ melden, das unsere Eigenkapital-Regel nicht erfasst (Eigenkapital fehlt bei 9,6 % der Positiven gegen 3,1 % der Negativen). `liquidity` fehlt in 11 % der Zeilen, meist weil keine kurzfristigen Verbindlichkeiten getrennt ausgewiesen sind; außerdem hat sie extreme Ausreißer (Minimum −235.000) bei Firmen mit fast null Bilanzsumme – das muss vor dem Modell durch Winsorisieren behandelt werden.

## 6. Was bleibt, wenn wir nur vollständige Zeilen behalten?

Ein Logit braucht für jede Zeile alle Variablen. Würde man jede Zeile mit einer Lücke streichen („complete cases“), bliebe Folgendes übrig:

In [8]:
complete = sample.dropna(subset=palepu)

def split_of(year):
    return "Train 2012-17" if year <= 2017 else ("Embargo 2018-19" if year <= 2019 else "Test 2020-22")

rows = []
for name, data in [("alle Zeilen", sample), ("nur vollständige", complete)]:
    for part in ["Train 2012-17", "Embargo 2018-19", "Test 2020-22", None]:
        d = data if part is None else data[data["fiscal_year"].map(split_of) == part]
        rows.append({"Daten": name, "Abschnitt": part or "gesamt", "Zeilen": len(d),
                     "Positive": int(d[Y].sum()), "positive Firmen": d.loc[d[Y] == 1, "cik"].nunique()})
display(pd.DataFrame(rows).pivot(index="Abschnitt", columns="Daten", values=["Zeilen", "Positive", "positive Firmen"])
          .loc[["Train 2012-17", "Embargo 2018-19", "Test 2020-22", "gesamt"]])

# Welche Variable kostet die meisten Zeilen?
print("Fehlende Werte je Variable (alle Zeilen):")
print(sample[palepu].isna().mean().sort_values(ascending=False).map("{:.1%}".format).to_string())

Zeilen                     Positive                   \
Daten           alle Zeilen nur vollständige alle Zeilen nur vollständige   
Abschnitt                                                                   
Train 2012-17         18058             9901         167               96   
Embargo 2018-19        5774             3856          45               32   
Test 2020-22           9596             6268          70               51   
gesamt                33428            20025         282              179   

                positive Firmen                   
Daten               alle Zeilen nur vollständige  
Abschnitt                                         
Train 2012-17                76               50  
Embargo 2018-19              32               25  
Test 2020-22                 43               33  
gesamt                      119               87

Fehlende Werte je Variable (alle Zeilen):
grdummy              33.6%
sales_growth_1y      25.7%
roe                  11.8%
liquidity            11.3%
float_to_book         7.3%
float_to_earnings     5.2%
leverage              2.2%
log_assets            0.8%
idummy                0.1%


**Ergebnis:** Nur mit vollständigen Zeilen blieben 20.025 von 33.428 Zeilen und **179 von 282 Positiven** – rund ein Drittel der Ziele ginge verloren, im Training bleiben 96 Positive von 50 Firmen. Der größte Verursacher ist `sales_growth_1y` (und damit `grdummy`): Für 2012 gibt es kein Vorjahr im Datensatz, und rund 15 % der Firmen melden keinen Umsatz-Tag. Bevor wir schätzen, müssen wir also entscheiden, wie wir mit fehlenden Werten umgehen (streichen oder auffüllen mit Fehlend-Indikator wie im alten Code).

## Kernergebnisse

1. Alle neun Palepu-Variablen sind gebaut; ROE ist bei Eigenkapital ≤ 0 leer, Liquidität ist eine Näherung ohne Wertpapiere, und `idummy` nutzt nur Take-Privates aus dem Vorjahr, angekündigt laut `first_event_date`.
2. Spätere Ziele sind weniger profitabel, weniger liquide, höher verschuldet und niedriger bewertet – aber nicht kleiner als andere Firmen.
3. Die beiden Palepu-Dummies zeigen das Gegenteil seiner Befunde: Das Wachstums-Ressourcen-Ungleichgewicht ist bei Zielen seltener, Take-Private-Aktivität in der Branche im Vorjahr dagegen häufiger.
4. Fehlende Werte sind das größte praktische Problem: Nur mit vollständigen Zeilen fiele ein Drittel der Ziele weg (179 statt 282), vor allem wegen des Umsatzwachstums.
5. Zwei Datenpunkte für später: Partnerships haben kein erfasstes Eigenkapital, und 14 Events im relevanten Zeitraum haben keinen SIC-Code, sodass `idummy` dort leicht unterschätzt sein kann.